In [1]:
import qutip as qt
from qutip import tensor, basis
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.optimize import curve_fit
from itertools import product
from quantum_logical.trotterization import Trotterization

In [2]:
# creating gates
from quantum_logical.gate_extender import Gate_extender, Convert_levels
dim = 2
N = 5
# creating the set of cnots (this will operate between the g and e levels)
cnot1 = qt.cnot(N=5, control=0, target=1)
cnot2 = qt.cnot(N=5, control=0, target=2)

cnot3 = qt.cnot(N=5, control=0, target=3)
cnot4 = qt.cnot(N=5, control=1, target=3)
 
cnot5 = qt.cnot(N=5, control=1, target=4)
cnot6 = qt.cnot(N=5, control=2, target=4)

# the x_gate needs to be made in a qutrit gate and will involve conversion 
x_gate = qt.Qobj([[0, 1],[1, 0]])

# hadamard gate
hada = 1/np.sqrt(2) * qt.Qobj([[1,1],[1,-1]])

x_layer = tensor(tensor([x_gate] * 3), tensor([qt.qeye(dim)] * 2))
hada_layer = tensor(tensor([hada] * 3), tensor([qt.qeye(dim)] * 2))

C:\Users\girgi\AppData\Local\Temp\ipykernel_20656\4260229839.py:6: DeprecationWarning: Importing functions/classes of the qip submodule directly from the namespace qutip is deprecated. Please import them from the submodule instead, e.g.
from qutip.qip.operations import cnot
from qutip.qip.circuit import QubitCircuit

  cnot1 = qt.cnot(N=5, control=0, target=1)
C:\Users\girgi\AppData\Local\Temp\ipykernel_20656\4260229839.py:7: DeprecationWarning: Importing functions/classes of the qip submodule directly from the namespace qutip is deprecated. Please import them from the submodule instead, e.g.
from qutip.qip.operations import cnot
from qutip.qip.circuit import QubitCircuit

  cnot2 = qt.cnot(N=5, control=0, target=2)
C:\Users\girgi\AppData\Local\Temp\ipykernel_20656\4260229839.py:9: DeprecationWarning: Importing functions/classes of the qip submodule directly from the namespace qutip is deprecated. Please import them from the submodule instead, e.g.
from qutip.qip.operations import cnot

In [3]:
# building the expectation value calculator 
basis0 = qt.Qobj([[1],[0]])
basis1 = qt.Qobj([[0],[1]])
vectors_ = [basis0, basis1]
hada_vectors = vectors = [hada * i for i in vectors_]
vectors = [tensor(i,j,k) for i in vectors for j in vectors for k in vectors]
vectors2 = [tensor(i,j,k) for i in vectors_ for j in vectors_ for k in vectors_]

In [4]:
correction_z = (hada * x_gate * hada.dag())
correction_z

Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
Qobj data =
[[ 1.  0.]
 [ 0. -1.]]

In [5]:
# generating parameters and creating initial state
T1 = 30
T2 = 30

trotter_dt = .005

# initializing Trotterization
trotter = Trotterization(trotter_dt=trotter_dt, T1=T1, T2=T2, dim=dim, num_qubits=N, qudit="qubit")

In [6]:
# creating initial state
# state controls 
alpha = 0
beta = 1

# state
psi0 = tensor((alpha * basis(dim, 0) + beta * basis(dim, 1)).unit(), tensor([basis(dim, 0)] * (N-1)))
rho0 = psi0 * psi0.dag()
encoded_rho = rho_encoded = hada_layer  * cnot1 * rho0 * cnot1.dag() * hada_layer.dag()
rho_encoded_traced = qt.ptrace(rho_encoded, [0,1,2]) 
rho_encoded_traced # density matrix print confirmation

Quantum object: dims = [[2, 2, 2], [2, 2, 2]], shape = (8, 8), type = oper, isherm = True
Qobj data =
[[ 0.125  0.125 -0.125 -0.125 -0.125 -0.125  0.125  0.125]
 [ 0.125  0.125 -0.125 -0.125 -0.125 -0.125  0.125  0.125]
 [-0.125 -0.125  0.125  0.125  0.125  0.125 -0.125 -0.125]
 [-0.125 -0.125  0.125  0.125  0.125  0.125 -0.125 -0.125]
 [-0.125 -0.125  0.125  0.125  0.125  0.125 -0.125 -0.125]
 [-0.125 -0.125  0.125  0.125  0.125  0.125 -0.125 -0.125]
 [ 0.125  0.125 -0.125 -0.125 -0.125 -0.125  0.125  0.125]
 [ 0.125  0.125 -0.125 -0.125 -0.125 -0.125  0.125  0.125]]

In [7]:
vals = []
for vec in vectors:
    val = (vec.dag() * qt.ptrace(rho_encoded, [0,1,2]) * vec)[0][0][0]
    vals.append(val)
vals

[0j, 0j, 0j, 0j, 0j, 0j, (0.999999999999999+0j), 0j]

In [8]:
cnots = [cnot1, cnot2, cnot3, cnot4, cnot5, cnot6]

In [9]:
def sim_func(rho_encoded, T1, T2):
    trotter = Trotterization(trotter_dt=trotter_dt, T1=T1, T2=T2, dim=dim, num_qubits=N, qudit="qubit")

    states = []
    no_error_states = []

    cnot_time = 1

    # rho_encoded0 = trotter.apply(rho=rho_encoded, duration=1, unitary=[tensor([qt.qeye(dim)] * N)], choice_error="yes")
    gate_time = [cnot_time for _ in range(4)]

    rho_encoded = hada_layer * rho_encoded * hada_layer.dag()
    rho_encoded1 = hada_layer * encoded_rho * hada_layer.dag()
    for i in range(2,6):
        rho_evo = trotter.apply(rho=rho_encoded, duration=gate_time[i - 2], unitary=[cnots[i]], errors=True)
        rho_evo1 = trotter.apply(rho=rho_encoded1, duration=gate_time[i - 2], unitary=[cnots[i]], errors=False)
        states.extend(rho_evo)
        no_error_states.extend(rho_evo1)
        rho_encoded = rho_evo[-1]
        rho_encoded1 = rho_evo1[-1]
    rho_evo = hada_layer * rho_encoded * hada_layer.dag()
    rho_evo1 = hada_layer * rho_encoded1 * hada_layer.dag()
    states.append(rho_evo)
    no_error_states.append(rho_evo1)

    # recovery operation
    # measurement and recovery process 
    # projective measurement 
    options = [0,1]
    proj = [qt.tensor(qt.qeye(dim), qt.qeye(dim), qt.qeye(dim), (qt.tensor(qt.basis(dim, i), qt.basis(dim, j)) * (qt.tensor(qt.basis(dim, i), qt.basis(dim, j))).dag())) for i in options for j in options]

    # measurement outcome value 
    projection_results = [(proj * states[-1]).tr() for proj in proj]

    # correction_operators
    r00 = qt.tensor([qt.qeye(dim)] * 5)
    r01 = qt.tensor(qt.qeye(dim), qt.qeye(dim), correction_z, qt.tensor([qt.qeye(dim)] * 2))
    r10 = qt.tensor(correction_z, qt.tensor([qt.qeye(dim)] * 4))
    r11 = qt.tensor(qt.qeye(dim), correction_z, qt.qeye(dim), qt.tensor([qt.qeye(dim)] * 2))
    recovery_ops = [[r00], [r01], [r10], [r11]]

    rec = [[proj[0], r00], [proj[1], r01], [proj[2], r10], [proj[3], r11]]

    import random
    # print(projection_results)

    # result = random.choices(rec, projection_results)
    # print(projection_results)
    # print(result)
    logicals = [vectors[5], vectors[3], vectors[6], vectors[-1]]
    vals = []
    for vec in logicals:
        val = (vec.dag() * qt.ptrace(states[-1], [0,1,2]) * vec)[0][0][0]
        vals.append(val)
    uncorrectable_error = 1 - np.abs(sum(vals))
    correctable_error = np.abs(sum(vals))

    detectable_error = sum([projection_results[i] for i in [0,2,3]]) # this is my metric for logical error 
    
    # imperfect measurement 
    projected_states = []
    # measurement_duration = .005
    for i in range(len(proj)):
        projection_state = proj[i] * states[-1] * proj[i].dag()
        # projected_state = trotter.apply(projection_state, duration=measurement_duration, unitary=[tensor([qt.qeye(dim)] * N)], errors=False)
        projected_states.append(projection_state)

    # recovery 
    corrected_states = []
    recovery_duration = .025
    for i in range(len(recovery_ops)):
        corrected_state = trotter.apply(projected_states[i], duration=recovery_duration, unitary=recovery_ops[i], errors=True)
        corrected_states.append(corrected_state)

    for i in range(len(corrected_state)):
        new_state = sum([projection_results[j] * corrected_states[j][i] for j in range(len(projection_results))])
        states.append(new_state)
        no_error_states.append(no_error_states[-1])

    logicals = [vectors[-1]]
    vals = []
    for vec in logicals:
        val = (vec.dag() * qt.ptrace(states[-1], [0,1,2]) * vec)[0][0][0]
        vals.append(val)
    errors_after_correction = 1 - np.abs(sum(vals))

    return errors_after_correction, states, no_error_states, detectable_error, uncorrectable_error, correctable_error

In [10]:
iterations = 300
states = []
no_error_states = []
fid = []
errors_after_corrections = []
detectable_errors = []
uncorrectable_errors = []
correctable_errors = []
phs = []
t1_list = np.linspace(.1, 120, iterations)
for i in tqdm(range(iterations)):
    errors_after_correction, state, no_error, detectable_error, uncorrectable_error, correctable_error = sim_func(rho_encoded=rho_encoded, T1=t1_list[i], T2=t1_list[i])
    states.extend(state)
    no_error_states.extend(no_error)
    # phs.append(errors_after_correction)
    errors_after_corrections.append(errors_after_correction)
    detectable_errors.append(detectable_error)
    uncorrectable_errors.append(uncorrectable_error)
    correctable_errors.append(correctable_error)

100%|██████████| 300/300 [11:38<00:00,  2.33s/it]


In [11]:
correctable_errors

[1.4288411745435043e-07,
 0.04141814270633286,
 0.1701841717371872,
 0.2933136174703926,
 0.3913576685447218,
 0.46787812544136986,
 0.5283055557193851,
 0.5768879649101301,
 0.6166505841232839,
 0.6497255869867208,
 0.6776331788206446,
 0.7014766148517606,
 0.7220717448443602,
 0.7400328902426079,
 0.7558304470882417,
 0.7698301665173668,
 0.7823204200993245,
 0.7935314594831636,
 0.8036492499704213,
 0.812825563199465,
 0.8211854482882163,
 0.8288328372825536,
 0.8358548034609479,
 0.8423248336123789,
 0.848305369628804,
 0.8538498020733433,
 0.8590040483683896,
 0.8638078128550859,
 0.8682956008329137,
 0.8724975407671733,
 0.8764400555903781,
 0.8801464143541657,
 0.8836371883686489,
 0.8869306305405907,
 0.8900429925701583,
 0.8929887914789605,
 0.8957810347962659,
 0.898431411510616,
 0.9009504548328522,
 0.9033476814615138,
 0.9056317111748362,
 0.9078103699727081,
 0.9098907793132336,
 0.9118794335832641,
 0.913782267546316,
 0.9156047153479254,
 0.9173517621561076,
 0.91902798

In [12]:
import csv


# Writing the arrays to a CSV file
with open('multiple_arrays_qubit.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(errors_after_corrections)  # Save array1 as a row
    writer.writerow(detectable_errors)  # Save array2 as a row
    writer.writerow(uncorrectable_errors)  # Save array1 as a row
    writer.writerow(correctable_errors)  # Save array2 as a row

print("Multiple arrays have been saved to multiple_arrays.csv")


Multiple arrays have been saved to multiple_arrays.csv
